In [ ]:
import pandas as pd
import json
import os

In [ ]:
# Directory where your final 'enriched' JSON files are located.
INPUT_DIR = "labeled_data_enriched"
# Optional: Filename to save the final dataset for later use.
OUTPUT_CSV_FILE = "final_training_dataset.csv"

In [ ]:
def build_dataset_from_json(input_dir):
    """
    Reads all enriched JSON files from a directory and compiles them
    into a single Pandas DataFrame.
    """
    all_spans = []
    
    if not os.path.exists(input_dir):
        print(f"❌ Error: Input directory '{input_dir}' not found.")
        print("Please make sure you have run the Stage 2 script and the directory exists.")
        return None

    print(f"📁 Reading enriched data from '{input_dir}'...")
    
    # Iterate through each file in the enriched data directory
    for filename in sorted(os.listdir(input_dir)):
        if filename.endswith(".json"):
            file_path = os.path.join(input_dir, filename)
            with open(file_path, 'r', encoding='utf-8') as f:
                spans_from_file = json.load(f)
                all_spans.extend(spans_from_file)
    
    if not all_spans:
        print("⚠️ Warning: No spans were loaded. The dataset is empty.")
        return None
        
    print(f"✅ Successfully loaded a total of {len(all_spans)} spans from {len(os.listdir(input_dir))} files.")
    
    # Create the main DataFrame from the list of span dictionaries
    df = pd.DataFrame(all_spans)
    
    return df


In [ ]:
def main():
    """
    Main function to execute Phase 1: Data Aggregation & Preparation.
    """
    # Build the full DataFrame
    master_df = build_dataset_from_json(INPUT_DIR)
    
    if master_df is None:
        return

    # --- Separate Features (X) and Labels (y) ---

    # The 'features' column currently contains lists. We need to expand it into
    # separate columns for the model.
    feature_columns = [f"feature_{i}" for i in range(len(master_df['features'].iloc[0]))]
    X = pd.DataFrame(master_df['features'].tolist(), columns=feature_columns)

    # The 'label' column is our target variable.
    y = master_df['label']

    print("\n--- Dataset Ready for Training ---")
    print(f"Features shape (X): {X.shape}")
    print(f"Labels shape (y):   {y.shape}")
    
    print("\nFirst 5 rows of features (X):")
    print(X.head())
    
    print("\nFirst 5 labels (y):")
    print(y.head())

    # --- Save the combined dataset for easy access later ---
    # We combine X and y for saving to a single CSV file.
    final_dataset = X.copy()
    final_dataset['label'] = y
    final_dataset.to_csv(OUTPUT_CSV_FILE, index=False)
    print(f"\n💾 Full dataset saved to '{OUTPUT_CSV_FILE}' for convenience.")
    return X,y


In [ ]:
if __name__ == "__main__":
    X, y = main()

In [ ]:
frequency_map = y.value_counts()
print(frequency_map)

In [ ]:
print(X.columns.tolist())